# 09 — Confounder Analysis

Can donor demographics (SEX, AGE, RACE, Hardy Scale, Ischemic Time) alone predict
tissue pathology? And does adding them to expression improve performance?

- **Model A** — RF with confounders only
- **Model B** — RF with expression + confounders
- **Expression-only RF** — from notebook 07

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from gtex_biomarkers.config import Config
from gtex_biomarkers.data import (
    load_cache, variance_filter,
    build_confounder_matrix,
)
from gtex_biomarkers.labels import discover_tissue_category_pairs
from gtex_biomarkers.models import make_rf_model
from gtex_biomarkers.utils import run_all_confounder_models_parallel

Config.ensure_dirs()

# Load cached data (run notebook 01 first)
X_wb, blood_subjid, _, df_meta_url, df_age = load_cache()

## 9.1 — Build Confounder Matrix

Map donor-level demographics (SEX, AGE, RACE, DTHHRDY, TRISCHD) to each blood sample.
RACE codes 98/99 (unknown) are treated as NaN and median-imputed.

In [2]:
X_conf = build_confounder_matrix(df_age, blood_subjid)
print(f"Confounder matrix: {X_conf.shape[0]} samples × {X_conf.shape[1]} features")
print(f"Features: {list(X_conf.columns)}")
print()
display(X_conf.describe().round(2))
display(X_conf.head(10))

Confounder matrix: 803 samples × 5 features
Features: ['SEX', 'AGE', 'RACE', 'DTHHRDY', 'TRISCHD']



,SEX,AGE,RACE,DTHHRDY,TRISCHD
count,803.00,803.00,803.00,803.00,803.00
mean,1.33,52.81,2.85,1.12,491.91
std,0.47,13.03,0.40,1.40,410.54
min,1.00,20.00,1.00,0.00,0.00
25%,1.00,47.00,3.00,0.00,79.00
50%,1.00,55.00,3.00,0.00,447.00
75%,2.00,63.00,3.00,2.00,824.50
max,2.00,70.00,4.00,4.00,1641.00


,SEX,AGE,RACE,DTHHRDY,TRISCHD
GTEX-1117F-0005-SM-HL9SH,2,66,2.0,4.0,1200.0
GTEX-111CU-0005-SM-GJ3PH,1,57,3.0,0.0,43.0
GTEX-111FC-0006-SM-H65Z1,1,61,3.0,1.0,1028.0
GTEX-111YS-0006-SM-5NQBE,1,62,3.0,0.0,74.0
GTEX-1122O-0005-SM-5O99J,2,64,3.0,0.0,35.0
GTEX-1128S-0005-SM-5P9HI,2,66,3.0,2.0,816.0
GTEX-113IC-0006-SM-5NQ9C,1,66,2.0,0.0,94.0
GTEX-113JC-0006-SM-5O997,2,53,3.0,2.0,611.0
GTEX-117XS-0005-SM-5PNU6,1,64,3.0,2.0,848.0
GTEX-117YW-0005-SM-5NQ8Z,1,58,3.0,3.0,785.0


## 9.2 — Discover Tissue × Category Pairs

In [4]:
pairs_df = discover_tissue_category_pairs(df_meta_url)
print(f"Total tissue × category pairs (≥ {Config.ALL_TISSUE_THRESHOLD} samples): {len(pairs_df)}")
display(pairs_df)

Total tissue × category pairs (≥ 50 samples): 53


,tissue,category,n_samples
0,Adipose - Subcutaneous,fibrosis,137
1,Adipose - Visceral (Omentum),fibrosis,99
2,Artery - Aorta,atherosis,221
3,Artery - Aorta,atherosclerosis,101
4,Artery - Aorta,sclerotic,64
5,Artery - Aorta,calcification,53
6,Artery - Coronary,calcification,158
7,Artery - Coronary,atherosclerosis,153
8,Artery - Coronary,sclerotic,124
9,Artery - Coronary,atherosis,116


## 9.3 — Run Models (Parallelized by Tissue)

For each tissue × category pair, train two RF models in a single pass:
- **Model A**: Confounders only (SEX, AGE, RACE, DTHHRDY, TRISCHD) — no feature selection
- **Model B**: Variance-filtered expression (top 100 per fold) + all confounders

In [5]:
# Variance-filtered expression (top 20K genes) — same as notebook 07
X_wb_var, _ = variance_filter(X_wb)
print(f"Variance-filtered expression: {X_wb_var.shape[0]} samples × {X_wb_var.shape[1]} genes")

conf_results, conf_summary, comb_results, comb_summary = \
    run_all_confounder_models_parallel(
        pairs_df, df_meta_url, blood_subjid,
        X_wb_var, X_conf, make_rf_model
    )

print(f"Completed {len(conf_results)} confounder-only models")
print(f"Completed {len(comb_results)} expression+confounder models")

Variance-filtered expression: 803 samples × 20000 genes


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done   6 out of  19 | elapsed:  6.5min remaining: 14.1min
[Parallel(n_jobs=-1)]: Done   8 out of  19 | elapsed:  6.6min remaining:  9.0min
[Parallel(n_jobs=-1)]: Done  10 out of  19 | elapsed:  7.9min remaining:  7.1min
[Parallel(n_jobs=-1)]: Done  12 out of  19 | elapsed:  8.3min remaining:  4.9min
[Parallel(n_jobs=-1)]: Done  14 out of  19 | elapsed:  8.4min remaining:  3.0min
[Parallel(n_jobs=-1)]: Done  16 out of  19 | elapsed:  9.5min remaining:  1.8min


Completed 53 confounder-only models
Completed 53 expression+confounder models


[Parallel(n_jobs=-1)]: Done  19 out of  19 | elapsed: 13.3min finished


## 9.4 — Summary Tables

In [6]:
# Save summaries
conf_summary.to_csv(Config.TABLES_DIR / "cv_results_confounder_only.csv", index=False)
comb_summary.to_csv(Config.TABLES_DIR / "cv_results_expr_plus_confounder.csv", index=False)

print("=== Model A: Confounder-Only RF ===")
display(conf_summary)
print()
print("=== Model B: Expression + Confounder RF ===")
display(comb_summary)

=== Model A: Confounder-Only RF ===


,tissue,category,mean_auc,std_auc,optimal_threshold
15,Breast - Mammary Tissue,gynecomastoid,0.868507,0.060523,0.296000
18,Breast - Mammary Tissue,atrophy,0.772285,0.039881,0.066000
16,Breast - Mammary Tissue,hyperplasia,0.754917,0.051751,0.326000
47,Spleen,congestion,0.720674,0.053307,0.982000
44,Pancreas,saponification,0.717051,0.024563,0.352000
48,Testis,spermatogenesis,0.650235,0.122915,0.952000
37,Lung,pneumonia,0.640349,0.074096,0.088000
10,Artery - Tibial,calcification,0.622944,0.052504,0.246000
19,Esophagus - Mucosa,esophagitis,0.619971,0.052468,0.247149
40,Muscle - Skeletal,atrophy,0.618750,0.092139,0.980000



=== Model B: Expression + Confounder RF ===


,tissue,category,mean_auc,std_auc,optimal_threshold
15,Breast - Mammary Tissue,gynecomastoid,0.875403,0.061324,0.360
18,Breast - Mammary Tissue,atrophy,0.844138,0.022329,0.056
47,Spleen,congestion,0.816906,0.026039,0.838
30,Liver,cirrhosis,0.794705,0.096380,0.150
16,Breast - Mammary Tissue,hyperplasia,0.751471,0.082132,0.362
37,Lung,pneumonia,0.741729,0.021265,0.128
44,Pancreas,saponification,0.731118,0.026903,0.456
48,Testis,spermatogenesis,0.719460,0.085770,0.916
40,Muscle - Skeletal,atrophy,0.691667,0.093541,0.844
28,Liver,congestion,0.683816,0.024515,0.620


## 9.5 — Three-Way Comparison

Merge confounder-only, expression-only (notebook 07), and expression+confounder results.
Compute deltas to see what each feature set contributes.

In [ ]:
# Load expression-only RF results from notebook 07
expr_only_path = Config.TABLES_DIR / "cv_results_all_tissue_rf.csv"
if not expr_only_path.exists():
    raise FileNotFoundError("Run notebook 07 first to generate expression-only RF results.")

expr_summary = pd.read_csv(expr_only_path)

# Three-way merge
three_way = (
    conf_summary[["tissue", "category", "mean_auc"]]
    .rename(columns={"mean_auc": "auc_conf"})
    .merge(
        expr_summary[["tissue", "category", "mean_auc"]]
        .rename(columns={"mean_auc": "auc_expr"}),
        on=["tissue", "category"], how="inner",
    )
    .merge(
        comb_summary[["tissue", "category", "mean_auc"]]
        .rename(columns={"mean_auc": "auc_comb"}),
        on=["tissue", "category"], how="inner",
    )
)
three_way["delta_comb_vs_expr"] = three_way["auc_comb"] - three_way["auc_expr"]
three_way = three_way.sort_values("auc_comb", ascending=False)
three_way.to_csv(Config.TABLES_DIR / "cv_three_way_comparison.csv", index=False)

print(f"Models where adding confounders helps (delta > 0): "
      f"{(three_way['delta_comb_vs_expr'] > 0).sum()}/{len(three_way)}")
print(f"Mean delta (Expr+Conf vs Expr-only): {three_way['delta_comb_vs_expr'].mean():.4f}")
display(three_way)

## 9.6 — Figures

In [ ]:
# 9.6a — Grouped bar chart: three models side by side
tw = three_way.copy()
tw["label"] = tw["tissue"] + " | " + tw["category"]
tw = tw.sort_values("auc_comb", ascending=True)

y_pos = np.arange(len(tw))
bar_h = 0.25

fig, ax = plt.subplots(figsize=(12, max(7, len(tw) * 0.55)))
ax.barh(y_pos - bar_h, tw["auc_conf"], bar_h, label="Confounder-only",
        color="#e74c3c", alpha=0.8)
ax.barh(y_pos, tw["auc_expr"], bar_h, label="Expression-only",
        color="#3498db", alpha=0.8)
ax.barh(y_pos + bar_h, tw["auc_comb"], bar_h, label="Expr + Confounder",
        color="#2ecc71", alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(tw["label"], fontsize=7)
ax.axvline(0.5, ls=":", color="grey", lw=0.8)
ax.set_xlabel("Mean ROC-AUC (5-fold CV)", fontsize=11)
ax.set_title("Three-Way Model Comparison", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0.3, 1.0)
fig.tight_layout()
fig.savefig(Config.FIGURES_DIR / "three_way_comparison_barplot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 9.6b — Delta bar chart: AUC gain from adding confounders to expression
from matplotlib.patches import Patch

s = three_way.sort_values("delta_comb_vs_expr", ascending=True).copy()
s["label"] = s["tissue"] + " | " + s["category"]
colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in s["delta_comb_vs_expr"]]

fig, ax = plt.subplots(figsize=(10, max(6, len(s) * 0.35)))
ax.barh(range(len(s)), s["delta_comb_vs_expr"], color=colors, alpha=0.85)
ax.set_yticks(range(len(s)))
ax.set_yticklabels(s["label"], fontsize=7)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("AUC delta (Expr+Conf − Expr-only)")
ax.set_title("AUC Change from Adding Confounders to Expression", fontsize=12)
ax.legend(
    handles=[Patch(color="#2ecc71", label="Confounders help"),
             Patch(color="#e74c3c", label="Confounders hurt")],
    loc="lower right", fontsize=9,
)
fig.tight_layout()
fig.savefig(Config.FIGURES_DIR / "confounder_delta_barplot.pdf", bbox_inches="tight")
plt.show()